In [1]:
import os
import sys
from pathlib import Path
from types import SimpleNamespace
import pandas as pd

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-21.0.10.7-hotspot\bin"

root_dir = Path.cwd().parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

from src.retrieval.build_bm25_index import (
    build_bm25_index_from_parquet_dir_batched,
    list_parquet_files,
    load_parquet_as_chunks,
    build_bm25_index_from_jsonl_tmp_dir,
)
from src.retrieval.retriever import BM25Retriever

## 1. parquet shards 경로 확인

In [2]:
PARQUET_DIR = root_dir / "data" / "interim" / "normalized_paragraphs" / "notebook_all_shards" / "shards"
BM25_INDEX_PATH = root_dir / "data" / "interim" / "bm25" / "bm25_index_all_shards_lucene"

parquet_files = list_parquet_files(PARQUET_DIR)
print("num parquet shards:", len(parquet_files))
print("first shard:", parquet_files[0])
print("last shard :", parquet_files[-1])

num parquet shards: 15517
first shard: d:\AI\projects\multi-hop-rag-hover\data\interim\normalized_paragraphs\notebook_all_shards\shards\AA__wiki_00.parquet
last shard : d:\AI\projects\multi-hop-rag-hover\data\interim\normalized_paragraphs\notebook_all_shards\shards\FZ__wiki_16.parquet


## 2. 샘플 shard 로딩 확인

In [3]:
sample_parquet = parquet_files[0]
df_sample = pd.read_parquet(sample_parquet)
print("sample parquet:", sample_parquet)
print("sample shape  :", df_sample.shape)
df_sample.head(3)

sample parquet: d:\AI\projects\multi-hop-rag-hover\data\interim\normalized_paragraphs\notebook_all_shards\shards\AA__wiki_00.parquet
sample shape  : (1476, 10)


,doc_id,title,title_norm,paragraph_id,paragraph_uid,paragraph_index,paragraph_text_raw,paragraph_text,source_path,record_id
0,12,Anarchism,anarchism,anarchism::p0000,962438bf4f264ce61278f51e83238d70573833fc6a69c0...,0,"Anarchism is a <a href=""political%20philosophy...",Anarchism is a political philosophy that advoc...,D:\AI\projects\multi-hop-rag-hover\data\raw\wi...,0
1,12,Anarchism,anarchism,anarchism::p0001,7b363efbdcb9f52afc35ab2e6101fbfab3958ce7864fe3...,1,"While <a href=""anti-statism"">anti-statism</a> ...","While anti-statism is central, anarchism speci...",D:\AI\projects\multi-hop-rag-hover\data\raw\wi...,0
2,12,Anarchism,anarchism,anarchism::p0002,185f03e40bc18222e4784ff86818a48ce736051ad2cc45...,2,Anarchism does not offer a fixed body of doctr...,Anarchism does not offer a fixed body of doctr...,D:\AI\projects\multi-hop-rag-hover\data\raw\wi...,0


## 3. 샘플 chunk 포맷 확인

In [4]:
sample_chunks = load_parquet_as_chunks(sample_parquet)
print("num chunks in sample shard:", len(sample_chunks))
print("=" * 100)
print(sample_chunks[0])

num chunks in sample shard: 1476
{'id': '962438bf4f264ce61278f51e83238d70573833fc6a69c0710cf1bc142983f4ba', 'text': 'Anarchism\nAnarchism is a political philosophy that advocates self-governed societies based on voluntary institutions. These are often described as stateless societies, although several authors have defined them more specifically as institutions based on non-hierarchical free associations. Anarchism holds the state to be undesirable, unnecessary and harmful.', 'metadata': {'doc_id': '12', 'title': 'Anarchism', 'title_norm': 'anarchism', 'paragraph_id': 'anarchism::p0000', 'paragraph_index': 0, 'source_path': 'D:\\AI\\projects\\multi-hop-rag-hover\\data\\raw\\wiki_2017\\AA\\wiki_00.bz2', 'record_id': 0}}


## 4. 전체 shard로 BM25 인덱스 생성 (batched loading)

In [ ]:
files_per_batch = 128  # ??? ??? ?? 32/64/128/256 ??
threads = 8            # CPU ??? ?? 4~8 ??

# ?? 1?(?? JSONL? ?? ?)? ?? ??
build_bm25_index_from_parquet_dir_batched(
    parquet_dir=PARQUET_DIR,
    output_path=BM25_INDEX_PATH,
    k1=1.5,
    b=0.75,
    files_per_batch=files_per_batch,
    use_tqdm=True,
    threads=threads,
)


Loading parquet directory in batches: d:\AI\projects\multi-hop-rag-hover\data\interim\normalized_paragraphs\notebook_all_shards\shards
files_per_batch: 128
[1/15517] Loading parquet: d:\AI\projects\multi-hop-rag-hover\data\interim\normalized_paragraphs\notebook_all_shards\shards\AA__wiki_00.parquet
[2/15517] Loading parquet: d:\AI\projects\multi-hop-rag-hover\data\interim\normalized_paragraphs\notebook_all_shards\shards\AA__wiki_01.parquet
[3/15517] Loading parquet: d:\AI\projects\multi-hop-rag-hover\data\interim\normalized_paragraphs\notebook_all_shards\shards\AA__wiki_02.parquet
[4/15517] Loading parquet: d:\AI\projects\multi-hop-rag-hover\data\interim\normalized_paragraphs\notebook_all_shards\shards\AA__wiki_03.parquet
[5/15517] Loading parquet: d:\AI\projects\multi-hop-rag-hover\data\interim\normalized_paragraphs\notebook_all_shards\shards\AA__wiki_04.parquet
[6/15517] Loading parquet: d:\AI\projects\multi-hop-rag-hover\data\interim\normalized_paragraphs\notebook_all_shards\shards\

In [4]:
jsonl_tmp_dir = root_dir / "data" / "interim" / "bm25" / "bm25_index_all_shards_lucene_jsonl_tmp"
output_index_dir = root_dir / "data" / "interim" / "bm25" / "bm25_index_all_shards_lucene_v2"

# JSONL?? ?? Lucene ??? ??? (parquet ??? ??)
build_bm25_index_from_jsonl_tmp_dir(
    jsonl_tmp_dir=jsonl_tmp_dir,
    output_path=output_index_dir,
    k1=1.5,
    b=0.75,
    threads=8,
)


JSONL files found: 122
first: part_00001.jsonl
last : part_00122.jsonl
[BM25/Pyserini] Building Lucene index at: d:\AI\projects\multi-hop-rag-hover\data\interim\bm25\bm25_index_all_shards_lucene


d:\AI\projects\multi-hop-rag-hover\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[BM25/Pyserini] Index loaded from d:\AI\projects\multi-hop-rag-hover\data\interim\bm25\bm25_index_all_shards_lucene
BM25 index built: d:\AI\projects\multi-hop-rag-hover\data\interim\bm25\bm25_index_all_shards_lucene


In [4]:
print("BM25 index exists:", BM25_INDEX_PATH.exists())
print("BM25 index path  :", BM25_INDEX_PATH)

BM25 index exists: True
BM25 index path  : d:\AI\projects\multi-hop-rag-hover\data\interim\bm25\bm25_index_all_shards_lucene


## 5. BM25Retriever 로 검색 테스트

In [5]:
config = SimpleNamespace()
config.data_dir = root_dir / "data"
config.bm25_index_path = output_index_dir
config.bm25_k1 = 1.5
config.bm25_b = 0.75
config.bm25_threads = 8

retriever = BM25Retriever(config=config, top_k=5)
retriever


d:\AI\projects\multi-hop-rag-hover\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[BM25/Pyserini] Index loaded from d:\AI\projects\multi-hop-rag-hover\data\interim\bm25\bm25_index_all_shards_lucene


In [6]:
queries = [
    "Anarchism is a political philosophy",
    "anti-statism is central to anarchism",
    "far-left ideology and anarchist economics",
]

for query in queries:
    print("\n" + "#" * 120)
    print("QUERY:", query)

    results = retriever.retrieve_for_claim(query, top_k=3)
    for i, r in enumerate(results, start=1):
        print("-" * 100)
        print("rank:", i)
        print("title:", r["metadata"].get("title"))
        print("paragraph_id:", r["metadata"].get("paragraph_id"))
        print("score:", r["score"])
        print("text:", r["text"][:300])



########################################################################################################################
QUERY: Anarchism is a political philosophy
----------------------------------------------------------------------------------------------------
rank: 1
title: Libertarian socialism
paragraph_id: libertarian socialism::p0003
score: 11.877900123596191
text: Past and present political philosophies and movements commonly described as libertarian socialist include [[anarchism]], as well as [[autonomism]], [[Communalism (political philosophy)|communalism]], [[participism]], [[guild socialism]], [[revolutionary syndicalism]] and [[libertarian Marxist]] phil
----------------------------------------------------------------------------------------------------
rank: 2
title: Lawlessness
paragraph_id: lawlessness::p0003
score: 11.460100173950195
text: Anarchism is a political philosophy that advocates self-governed societies based on voluntary institutions.
--------------------